# Typed findings — `response_schema`

Day 1 returned `reply.body` — a markdown string. Useful, but the moment you want to do anything programmatic with the result (index it, persist it, hand it to another agent), you're back to parsing prose. Today the agent returns a Pydantic object instead.

Make sure `OPENAI_API_KEY` is in your `.env`.

In [1]:
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

load_dotenv()

config = OpenAIConfig(
    model="gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.openai.com/v1",
)

## The schema

`Field(description=...)` strings are not decoration — they go into the JSON schema the model sees, so they double as instructions. `Field(ge=0.0, le=1.0)` becomes a JSON-schema constraint that gets validated on the way back.

Heads-up on strict mode: every field has to be required, so no Pydantic defaults below. (Need defaults? Wrap with `PromptedSchema(...)` to skip strict mode.)

In [2]:
class Citation(BaseModel):
    title: str = Field(description="The cited work or source.")
    relevance: str = Field(description="One sentence on why this source supports the finding.")

class Finding(BaseModel):
    """One self-contained research result."""
    topic: str = Field(description="The specific aspect of the question this finding addresses.")
    summary: str = Field(description="2-3 sentences capturing the core mechanism or claim.")
    citations: list[Citation] = Field(description="Sources behind the summary; emit [] if none.")
    novelty: float = Field(ge=0.0, le=1.0, description="0 = textbook, 1 = cutting edge.")

## Ask with `response_schema`

Hand the agent the schema. The framework forwards the JSON schema to the model and validates on the way back. `reply.body` is still the raw string; `await reply.content()` is what gives you the typed instance.

In [3]:
from autogen.beta.tools import ExaToolkit
exa_tool = ExaToolkit(api_key = os.getenv("EXA_API_KEY"))
theorist = Agent(
    name="theorist",
    prompt="You're a careful theoretical physicist. Be specific.",
    config=config,
    response_schema=Finding,
    tools=[exa_tool]
)

reply = await theorist.ask("Produce drill innovatively into high-Tc superconductivity as a structured finding.")
finding: Finding = await reply.content()

print(type(finding).__name__, "|", finding.topic, "| novelty=", finding.novelty)

Finding | High-Tc superconductivity: mechanism and current theoretical status | novelty= 1.0


## Render

In [4]:
lines = [f"### {finding.topic} — novelty={finding.novelty:.2f}", "", finding.summary]
for c in finding.citations:
    lines.append(f"- **{c.title}** — {c.relevance}")
display(Markdown("\n".join(lines)))

### High-Tc superconductivity: mechanism and current theoretical status — novelty=1.00

The central theoretical issue in high-Tc superconductivity, especially in cuprates, is still the lack of a universally accepted microscopic mechanism that explains both pairing and the material-dependent Tc trends. Recent work increasingly supports strong-coupling, magnetically driven physics: short-range spin fluctuations, superexchange, and possibly resonant/Feshbach-like interactions between doped carriers and nearby bound states are prominent candidates, while phonons appear insufficient as a sole pairing glue in cuprates. At the same time, ab initio many-body approaches are beginning to reproduce empirically important trends such as pressure and layer dependence, suggesting that predictive, material-specific modeling is becoming feasible even if the full pairing mechanism remains unsettled.
- **High-temperature superconductivity | Nature Reviews Physics (2021)** — Summarizes the field’s open problem: no established microscopic theory and multiple competing ideas about unconventional pairing in high-Tc materials.
- **Ab initio quantum many-body description of superconducting trends in the cuprates (2025)** — Shows that first-principles many-body calculations can reproduce pressure and layer trends and points to superexchange and covalency as useful descriptors.
- **Feshbach hypothesis of high-Tc superconductivity in cuprates (2025)** — Proposes a strong-coupling resonance mechanism tied to doped Mott insulators and spin-polaron physics, illustrating one innovative route beyond conventional glue pictures.
- **Charge Correlations in Cuprate Superconductors (2024)** — Documents the ubiquity of charge-density-wave correlations and their competition/intertwining with superconductivity, a key part of the modern cuprate phase diagram.

## Up next

`Finding` is a contract for *one* agent's output. But there's nothing stopping you from making a schema that describes a *team* of agents — and an agent that emits one. Tomorrow: a planner that returns a typed `ResearchPlan`, and we spawn the workers it described.